# Colab runner
Select `Runtime > Change runtime type > T4 GPU` first.
Then run the cells from top to bottom (Shift+Enter).

The repository, including `outputs/02_corpus_v2` from step 02, is expected in Google Drive at `MyDrive/eeg-pooling-confound`.

In [ ]:
# 1) Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2) GPU check: a GPU (e.g. 'Tesla T4') should be listed
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 3) Repository folder in Drive
%cd /content/drive/MyDrive/eeg-pooling-confound
!ls
!ls -la outputs/02_corpus_v2

In [ ]:
# 4) Copy the corpus to the local disk of Colab (training from Drive is slow), ~1-2 min
!mkdir -p /content/corpus && cp outputs/02_corpus_v2/X_z.npy outputs/02_corpus_v2/meta.csv /content/corpus/
import os; os.environ['REV_CORPUS_DIR'] = '/content/corpus'
!ls -la /content/corpus

In [ ]:
# 5) STEP 4: pooled, 5 splits x 3 seeds (~25-40 min GPU)
!python 04_pooled_repeated.py

In [ ]:
# 6) STEP 5: LODO, 3 seeds (~30-45 min GPU)
!python 05_lodo_multiseed.py

In [ ]:
# 7) STEP 6: architectures + complexity (~60-90 min GPU)
!python 06_architectures.py

In [ ]:
# 8) STEP 7: positive control, arithmetic vs rest, SAM40 <-> MA (~10-15 min)
!python 07_positive_control.py

In [ ]:
# 9) STEP 8: dataset-adversarial training, 5 lambdas, source-validation AUC, identity probe (~40-60 min)
!python 08_adversarial_v2.py

## Steps 11, 12, 9, 3 (GPU), 10 and 16
In a new session, run cells 1, 3 and 4 first, then **cell 10a** (copies the raw corpus).

In [ ]:
# 10a) Copy the RAW corpus as well (needed by steps 9 and 10), ~1 min
!cp outputs/02_corpus_v2/X_raw.npy /content/corpus/ && ls -la /content/corpus

In [ ]:
# 11) STEP 11: dataset-prior / class-balance test (~10-15 min)
!python 11_balanced_prior.py

In [ ]:
# 12) STEP 12: SAM40 task-based labels (~15-20 min)
!python 12_sam40_task_labels.py

In [ ]:
# 13) STEP 9: Euclidean alignment (~20-30 min)
!python 09_euclidean_alignment.py

In [ ]:
# 14) STEP 3 on the GPU, so that all reported numbers come from the same hardware (~30-40 min)
!mv outputs/03_within outputs/03_within_cpu 2>/dev/null; python 03_within_dataset_cv.py

In [ ]:
# 15) STEP 10: normalisation sensitivity (~40-60 min)
!python 10_normalisation_sensitivity.py

In [ ]:
# 16) STEP 16: state-only LODO, 3 seeds (~10-15 min)
!python 16_state_only_lodo.py

**If the session disconnects:** run cells 1, 3 and 4 again, then the same step. Finished runs are skipped.

Results are written to `outputs/<step>/RESULTS.txt` in the Drive folder.